# Final CUDA/T4 V-14 0.5x core run

This notebook executes exactly the owner-authorized 0.5x plan `e5b44620206e5bbaca3a03e890a4f99b4f29a23b87d3e2d4f8f8be2c3067d71d`: all eight independent coordinate ranges on a Tesla T4. It writes only plan-bound per-coordinate records and manifests.

It performs no merge, aggregation, rate calculation, comparison, inference, or conclusion. Every batch is first sealed locally, then byte-verified while transferring to Drive; rerunning with the same label skips sealed final directories.

In [ ]:
import shutil
import subprocess

assert shutil.which('nvidia-smi'), 'No GPU backend: select a T4 runtime, reconnect, then rerun.'
assert shutil.which('nvcc'), 'nvcc is unavailable: this is not the required CUDA/T4 runtime.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True).strip()
assert gpu.startswith('Tesla T4,'), f'Required Tesla T4, got: {gpu}'
print('GPU:', gpu)
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import tarfile

EXPECTED_CAPSULE_SHA256 = 'ef0681f50165615cf9a862778f80ce65b58b2b643a7fe4d2612a16dc59b71337'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_final_2x_capsule.tar.gz
assert len(uploaded) == 1, 'Upload exactly one capsule.'
archive = Path(next(iter(uploaded)))
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print(f'{archive.name}: {actual}')
assert actual == EXPECTED_CAPSULE_SHA256, 'Wrong capsule; do not continue.'
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')
ROOT = Path('/content/project')
CONTROL_INPUT = Path('/content/g6_v14_2x_core_control')
assert (ROOT / 'research/g6/run_cuda_v14_core_batches.py').is_file() and CONTROL_INPUT.is_dir()
print('Capsule extracted:', ROOT)

In [ ]:
from google.colab import drive
import json
import os
import sys

sys.dont_write_bytecode = True
drive.mount('/content/gdrive')

PREP_LABEL = 'g6_cuda_t4_v14_core_0_5x_preparation_20260921'
RUN_LABEL = 'g6_cuda_t4_v14_core_0_5x_20260921'
PREP_DIR = Path('/content/gdrive/MyDrive') / PREP_LABEL
RUN_ROOT = Path('/content/gdrive/MyDrive') / RUN_LABEL
ARTIFACT_ROOT = RUN_ROOT / 'artifacts'
CONTROL_OUTPUT = RUN_ROOT / 'control'
LOCAL_ARTIFACT_ROOT = Path('/content/g6_v14_core_0_5x_local_stage')
assert PREP_DIR.is_dir(), f'0.5x preparation directory is missing: {PREP_DIR}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(exist_ok=True)
CONTROL_OUTPUT.mkdir(exist_ok=True)
LOCAL_ARTIFACT_ROOT.mkdir(exist_ok=True)

EXPECTED_PLAN_CONTENT_SHA256 = 'e5b44620206e5bbaca3a03e890a4f99b4f29a23b87d3e2d4f8f8be2c3067d71d'
EXPECTED_FILES = {
    'unbalanced_qualification.json': 'e3a95d24ea257da47c6ab8654ce137bc6ab1b35e19ac187cacd5ad2bda40b890',
    'slack_qualification.json': 'd60a90413fa7ed2ce83ce2f175ea7bddf7d9ad672ca1e7a9dbf5e93f3946278b',
}
def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
for name, wanted in EXPECTED_FILES.items():
    assert digest(CONTROL_INPUT / name) == wanted, name

sys.path.insert(0, str(ROOT))
from research.g6.cuda_qualification import cuda_v14_core_batch as core
from research.g6.cuda_qualification import v14_core_batch_contract as contract
plan_source = PREP_DIR / 'v14_0_5x_core_batch_plan.json'
assert plan_source.is_file(), f'Missing prepared 0.5x plan: {plan_source}'
plan = json.loads(plan_source.read_text(encoding='utf-8'))
assert plan['content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256 and plan['scale'] == '0.5x'
by_id = contract.validate_plan(plan)
assert len(by_id) == 8
assert contract.cuda.current_hashes(contract.PRODUCER_RELATIVES) == plan['source_sha256']

def write_once(path, payload):
    if path.exists() or path.is_symlink():
        assert path.is_file() and not path.is_symlink() and path.read_bytes() == payload, f'Existing control file differs: {path}'
        return
    descriptor = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o644)
    with os.fdopen(descriptor, 'wb') as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

for name in EXPECTED_FILES:
    write_once(CONTROL_OUTPUT / name, (CONTROL_INPUT / name).read_bytes())
write_once(CONTROL_OUTPUT / plan_source.name, plan_source.read_bytes())
release = {
    'schema': core.SCHEMA_RELEASE,
    'plan_content_sha256': EXPECTED_PLAN_CONTENT_SHA256,
    'released_by': 'Shivansh Srivastava',
    'date': '2026-09-21',
    'review_id': 'owner-final-cuda-t4-v14-0-5x-core-2026-09-21',
    'authorizes': 'run-batch',
}
release['content_sha256'] = core.content_sha256(release)
write_once(CONTROL_OUTPUT / 'release_record.json', core.canonical_bytes(release))
assert json.loads((CONTROL_OUTPUT / 'release_record.json').read_text()) == release
core.validate_release(release, plan)
print('0.5x plan and controls: PASS')
print('Plan:', plan['content_sha256'])
print('Release:', release['content_sha256'])
print('Drive artifacts:', ARTIFACT_ROOT)

In [ ]:
# Approved 0.5x producer: eight independent ranges, locally sealed and byte-verified on Drive.
import shutil
import time
import uuid

ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
unbalanced_record = CONTROL_OUTPUT / 'unbalanced_qualification.json'
slack_record = CONTROL_OUTPUT / 'slack_qualification.json'
plan_file = CONTROL_OUTPUT / 'v14_0_5x_core_batch_plan.json'
release_file = CONTROL_OUTPUT / 'release_record.json'
ordered_batches = sorted(plan['batches'], key=lambda batch: batch['index'])
assert len(ordered_batches) == 8

def sha256_file(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            hasher.update(block)
    return hasher.hexdigest()

def transfer_sealed_batch(local, remote):
    assert local.is_dir() and not local.is_symlink() and not remote.exists() and not remote.is_symlink()
    members = sorted(entry.name for entry in local.iterdir())
    assert members == ['manifest.json', 'records.json'], members
    local_manifest = json.loads((local / 'manifest.json').read_text(encoding='utf-8'))
    assert local_manifest['content_sha256'] == core.content_sha256(local_manifest)
    assert sha256_file(local / 'records.json') == local_manifest['records']['file_sha256']
    staging = ARTIFACT_ROOT / f'.pending-transfer-{local.name}-{uuid.uuid4().hex}'
    shutil.copytree(local, staging, copy_function=shutil.copyfile)
    assert sorted(entry.name for entry in staging.iterdir()) == members
    assert sha256_file(staging / 'records.json') == sha256_file(local / 'records.json')
    assert sha256_file(staging / 'manifest.json') == sha256_file(local / 'manifest.json')
    os.rename(staging, remote)
    assert sha256_file(remote / 'records.json') == local_manifest['records']['file_sha256']
    shutil.rmtree(local)

def resume_sealed_batch(remote, batch):
    assert remote.is_dir() and not remote.is_symlink()
    assert sorted(entry.name for entry in remote.iterdir()) == ['manifest.json', 'records.json']
    manifest = json.loads((remote / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['content_sha256'] == core.content_sha256(manifest)
    assert manifest['batch_id'] == batch['batch_id']
    assert manifest['plan_content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256
    assert manifest['range'] == {'start': batch['start'], 'stop': batch['stop'], 'count': batch['count']}
    assert manifest['records']['record_count'] == batch['count'] and (remote / 'records.json').is_file()

completed = 0
started = time.monotonic()
for ordinal, batch in enumerate(ordered_batches, start=1):
    batch_id = batch['batch_id']
    remote = ARTIFACT_ROOT / batch_id
    local = LOCAL_ARTIFACT_ROOT / batch_id
    if remote.exists() or remote.is_symlink():
        resume_sealed_batch(remote, batch)
        completed += 1
        print(f'[{ordinal}/8] already complete: {batch_id}', flush=True)
        continue
    if local.exists() or local.is_symlink():
        print(f'[{ordinal}/8] retrying Drive transfer: {batch_id}', flush=True)
        transfer_sealed_batch(local, remote)
    else:
        command = [
            'python3', 'research/g6/run_cuda_v14_core_batches.py', '--run-batch',
            '--unbalanced-record', str(unbalanced_record),
            '--slack-record', str(slack_record),
            '--plan-file', str(plan_file),
            '--batch-id', batch_id,
            '--release-record', str(release_file),
            '--output-dir', str(LOCAL_ARTIFACT_ROOT),
        ]
        print(f'[{ordinal}/8] running {batch_id}: [{batch["start"]}, {batch["stop"]})', flush=True)
        result = subprocess.run(command, cwd=ROOT, env=ENV, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(f'Batch {batch_id} failed; no later batch was started.\n\n{result.stderr[-8000:]}')
        receipt = json.loads(result.stdout)
        assert receipt['verdict'] == 'written' and receipt['batch_id'] == batch_id
        assert receipt['record_count'] == batch['count'] and Path(receipt['artifact_directory']) == local
        transfer_sealed_batch(local, remote)
    completed += 1
    print(f'[{ordinal}/8] written and transferred: {batch_id}; completed={completed}/8; elapsed={(time.monotonic() - started) / 60:.1f} min', flush=True)

assert completed == 8
print('All eight approved 0.5x batch artifacts are present. Stop here: no merge, aggregation, comparison, rate, or interpretation is performed.')